# Phase 1 — Lesson 13: Numerical Stability
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 13: Numerical Stability
**Type:** Build | **Languages:** Python | **Time:** ~120 minutes
**Prerequisites:** Phase 1, Lessons 01-04



# What You'll Learn

---

## Learning Objectives

- Implement numerically stable softmax and log-sum-exp using the max-subtraction trick
- Identify overflow, underflow, and catastrophic cancellation in floating-point computations
- Verify analytical gradients against numerical gradients using centered finite differences
- Explain why bfloat16 is preferred over float16 for training and how loss scaling prevents gradient underflow


# The Problem

---

Your model trains for three hours, then the loss becomes NaN. Every ML bug you will debug eventually comes down to floating point.


# IEEE 754: How Computers Store Real Numbers

---

| Format | Bits | Decimal digits | Range |
|--------|------|----------------|-------|
| float64 | 64 | ~15-16 | ±1.8e308 |
| float32 | 32 | ~7-8 | ±3.4e38 |
| float16 | 16 | ~3-4 | ±65,504 |
| bfloat16 | 16 | ~2-3 | ±3.4e38 |

float32 gives about 7 decimal digits. float16 gives about 3. bfloat16 keeps float32's range with less precision.


In [ ]:
import structdef float32_to_float16(x):    packed = struct.pack('f', x)    packed16 = struct.pack('e', struct.unpack('f', packed)[0])    return struct.unpack('e', packed16)[0]print(f"0.1 + 0.2 = {0.1 + 0.2}")print(f"0.1 + 0.2 == 0.3? {0.1 + 0.2 == 0.3}")print(f"float16 max: {float32_to_float16(70000.0)}")  # overflow to inf

# The Log-Sum-Exp Trick

---

Computing `log(sum(exp(x_i)))` directly overflows if any x_i is large.

The trick: subtract the maximum before exponentiating:
```
log(sum(exp(x_i))) = max(x) + log(sum(exp(x_i - max(x))))
```


In [ ]:
import mathdef logsumexp_naive(values):    return math.log(sum(math.exp(v) for v in values))def logsumexp_stable(values):    c = max(values)    return c + math.log(sum(math.exp(v - c) for v in values))safe = [1.0, 2.0, 3.0]print(f"Naive:  {logsumexp_naive(safe):.6f}")print(f"Stable: {logsumexp_stable(safe):.6f}")large = [500.0, 501.0, 502.0]print(f"Stable on large: {logsumexp_stable(large):.6f}")print(f"Naive on large: {logsumexp_naive(large)}")  # inf

In [ ]:
import mathdef softmax_naive(logits):    exps = [math.exp(z) for z in logits]    t = sum(exps)    return [e / t for e in exps]def softmax_stable(logits):    m = max(logits)    exps = [math.exp(z - m) for z in logits]    t = sum(exps)    return [e / t for e in exps]dangerous = [100.0, 0.0, 0.0]print(f"Naive on [100,0,0]: {softmax_naive(dangerous)}")  # nanprint(f"Stable on [100,0,0]: {softmax_stable(dangerous)}")  # [1, 0, 0]

In [ ]:
import matplotlib.pyplot as pltimport math# Show exp overflow boundaryx = [85, 86, 87, 88, 89, 90]vals = []for v in x:    try:        vals.append(math.exp(v))    except OverflowError:        vals.append(float('inf'))plt.figure(figsize=(7, 4))plt.semilogy(x, vals, 'ro-', lw=2, markersize=6)plt.axhline(3.4e38, color='#e74c3c', linestyle='--', label='float32 max')plt.xlabel('x'); plt.ylabel('exp(x)')plt.title('exp() Overflow in float32', fontsize=12, fontweight='bold')plt.legend(); plt.grid(True, alpha=0.3)plt.tight_layout(); plt.show()print(f"exp(88) = {math.exp(88):.2e}")print(f"exp(89) = {math.exp(89)}")  # inf

# Why bfloat16 Wins for Training

---

float16: 5 exponent bits (max ~65,504). bfloat16: 8 exponent bits (same range as float32).

For training: range matters more than precision. bfloat16 is a simple truncation of float32.


In [ ]:
import structdef simulate_bfloat16(x):    packed = struct.pack('f', x)    as_int = int.from_bytes(packed, 'little')    truncated = as_int & 0xFFFF0000    repacked = truncated.to_bytes(4, 'little')    return struct.unpack('f', repacked)[0]print(f"float32: 1.23456789 -> {1.23456789}")print(f"bfloat16: {simulate_bfloat16(1.23456789)}")print(f"float16 max: ~65504, bfloat16 max: ~3.4e38")

In [ ]:
import mathdef clip_by_norm(gradients, max_norm):    total = math.sqrt(sum(g**2 for g in gradients))    if total > max_norm:        scale = max_norm / total        return [g * scale for g in gradients]    return gradientsgrads = [10.0, 20.0, 30.0]clipped = clip_by_norm(grads, 5.0)print(f"Original norm: {math.sqrt(sum(g**2 for g in grads)):.2f}")print(f"Clipped norm:  {math.sqrt(sum(g**2 for g in clipped)):.2f}")

In [ ]:
import mathdef numerical_gradient(f, x, h=1e-5):    grad = []    for i in range(len(x)):        xp = x[:]; xm = x[:]        xp[i] += h; xm[i] -= h        grad.append((f(xp) - f(xm)) / (2 * h))    return graddef f(params):    x, y = params    return x**2 + 3*x*y + y**3def f_grad(params):    x, y = params    return [2*x + 3*y, 3*x + 3*y**2]point = [2.0, 1.0]ana = f_grad(point)num = numerical_gradient(f, point)for i, (a, n) in enumerate(zip(ana, num)):    denom = max(abs(a), abs(n), 1e-8)    rel = abs(a - n) / denom    print(f"param {i}: analytical={a:.8f} numerical={n:.8f} rel_err={rel:.2e}")

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** Why does softmax need max-subtraction?
> **Explanation:** exp(100) overflows float32 → inf → inf/inf = NaN.

**Q2:** Why does bfloat16 win for training?
> **Explanation:** Same range as float32 (max 3.4e38), less precision — range matters more for gradients.

**Q3:** What does gradient clipping prevent?
> **Explanation:** Exploding gradients from ruining weights in one step.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Implement stable sigmoid — 1/(1+exp(-x)) overflows for very negative x.

**Exercise 2:** Compute variance of [1e20, 1e20+1, 1e20] with naive formula vs Welford's algorithm.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Overflow | Number too big | Result exceeds max → inf |
| Underflow | Number too small | Result becomes 0 |
| Log-sum-exp trick | Subtract the max first | log(Σexp(x)) computed stably |
| bfloat16 | Brain float 16 | 8 exponent bits like float32, preferred for training |
| Gradient clipping | Cap the gradient norm | Prevents exploding gradients |


# Further Reading

---

- [What Every Computer Scientist Should Know About Floating-Point Arithmetic](https://docs.oracle.com/cd/E19957-01/806-3568/ncg_goldberg.html)
- [Mixed Precision Training (Micikevicius et al., 2018)](https://arxiv.org/abs/1710.03740)


# Next Steps

---

When you're ready, continue to Lesson 14: Norms and Distances.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
